

# 1. (How the fetching works)
Siphon, a library used to access THREDDS, a data server that provides web-based access to scientific datasets and allows specific data extraction. (So you don't have to download everything needlessley)

In [ ]:
%pip install siphon

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.1/65.1 kB 3.3 MB/s eta 0:00:00


Here we get and list available forecast files from 2015-01-15, The whole dataset for GFS 0.25 spans from 2015-01-15 to present

Go here if you want to browse the files yourself:

https://thredds.rda.ucar.edu/thredds/catalog/files/g/d084001/catalog.html

In [ ]:
from siphon.catalog import TDSCatalog
catalog = TDSCatalog('https://thredds.rda.ucar.edu/thredds/catalog/files/g/d084001/2015/20150115/catalog.xml')
print(catalog.datasets)

['gfs.0p25.2015011500.f000.grib2', 'gfs.0p25.2015011500.f003.grib2', 'gfs.0p25.2015011500.f006.grib2', 'gfs.0p25.2015011500.f009.grib2', 'gfs.0p25.2015011500.f012.grib2', 'gfs.0p25.2015011500.f015.grib2', 'gfs.0p25.2015011500.f018.grib2', 'gfs.0p25.2015011500.f021.grib2', 'gfs.0p25.2015011500.f024.grib2', 'gfs.0p25.2015011500.f027.grib2', 'gfs.0p25.2015011500.f030.grib2', 'gfs.0p25.2015011500.f033.grib2', 'gfs.0p25.2015011500.f036.grib2', 'gfs.0p25.2015011500.f039.grib2', 'gfs.0p25.2015011500.f042.grib2', 'gfs.0p25.2015011500.f045.grib2', 'gfs.0p25.2015011500.f048.grib2', 'gfs.0p25.2015011500.f051.grib2', 'gfs.0p25.2015011500.f054.grib2', 'gfs.0p25.2015011500.f057.grib2', 'gfs.0p25.2015011500.f060.grib2', 'gfs.0p25.2015011500.f063.grib2', 'gfs.0p25.2015011500.f066.grib2', 'gfs.0p25.2015011500.f069.grib2', 'gfs.0p25.2015011500.f072.grib2', 'gfs.0p25.2015011500.f075.grib2', 'gfs.0p25.2015011500.f078.grib2', 'gfs.0p25.2015011500.f081.grib2', 'gfs.0p25.2015011500.f084.grib2', 'gfs.0p25.201

**Understanding the file naming convention:**

`gfs.0p25.YYYYMMDDHH.fXXX.grib2`

Breaking It Down:

`gfs.0p25` → This refers to the GFS(Global Forecast System) model with 0.25-degree resolution (high resolution).

`YYYYMMDDHH` → The initialization time of the forecast in UTC. (REAL OBSERVATION STARTING POINT)

2015011500 → January 15, 2015, at 00:00 UTC. (02:00 SWEDISH SUMMER TIME)

2015011506 → January 15, 2015, at 06:00 UTC. (08:00 SWEDISH SUMMER TIME)

2015011512 → January 15, 2015, at 12:00 UTC. (14:00 SWEDISH SUMMER TIME)

2015011518 → January 15, 2015, at 18:00 UTC. (20:00 SWEDISH SUMMER TIME)

`fXXX` → The forecast hour.

f000 → This is the analysis time (the actual observations at initialization time).

f003 → 3-hour forecast from initialization.

f006 → 6-hour forecast from initialization.

f009 → 9-hour forecast from initialization.

…and so on, in 3-hour increments.

So, Let's extract the datasets spanning a 10 days prognosis, Meaning 240 hours onwards and start at 12UTC initialization.

In [ ]:
#list all datasets
all_datasets = list(catalog.datasets.keys())

init_time = "2015011512"  # (12 UTC) init
prognosis = 168 #forecast hours <= 240 (interval of 3 hours) max is 384 (16 days)
#filter datasets that have "2015011512" (12 UTC) and forecast hours <= 240
filtered_files = [f for f in all_datasets if init_time in f and int(f.split(".f")[-1][:3]) <= prognosis]

#print the selected files
print("Selected files for processing:", filtered_files)

Selected files for processing: ['gfs.0p25.2015011512.f000.grib2', 'gfs.0p25.2015011512.f003.grib2', 'gfs.0p25.2015011512.f006.grib2', 'gfs.0p25.2015011512.f009.grib2', 'gfs.0p25.2015011512.f012.grib2', 'gfs.0p25.2015011512.f015.grib2', 'gfs.0p25.2015011512.f018.grib2', 'gfs.0p25.2015011512.f021.grib2', 'gfs.0p25.2015011512.f024.grib2', 'gfs.0p25.2015011512.f027.grib2', 'gfs.0p25.2015011512.f030.grib2', 'gfs.0p25.2015011512.f033.grib2', 'gfs.0p25.2015011512.f036.grib2', 'gfs.0p25.2015011512.f039.grib2', 'gfs.0p25.2015011512.f042.grib2', 'gfs.0p25.2015011512.f045.grib2', 'gfs.0p25.2015011512.f048.grib2', 'gfs.0p25.2015011512.f051.grib2', 'gfs.0p25.2015011512.f054.grib2', 'gfs.0p25.2015011512.f057.grib2', 'gfs.0p25.2015011512.f060.grib2', 'gfs.0p25.2015011512.f063.grib2', 'gfs.0p25.2015011512.f066.grib2', 'gfs.0p25.2015011512.f069.grib2', 'gfs.0p25.2015011512.f072.grib2', 'gfs.0p25.2015011512.f075.grib2', 'gfs.0p25.2015011512.f078.grib2', 'gfs.0p25.2015011512.f081.grib2', 'gfs.0p25.201501

siphon and THREDDS allows us to query for subset regions, So let's define a subset region for Sweden

In [ ]:
#define Sweden's bounding box
lat_min, lat_max = 55.0, 70.0
lon_min, lon_max = 10.0, 25.0

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive



# Fetching the Data
**PREREQUISITIES:**


1. `unique_dates.json` UNIQUE GRADING DATES (FILTERED BY HÖSTVETE & BLADFLÄCKSVAMPAR WITH DATES > 2015)

2. `proxies.json` public IP addresses(proxies) to avoid rate limiting

Can be found here on [Google Drive](https://drive.google.com/drive/folders/10BE1MqYS6C0GrExK3MrMJqkY0C2szWOg)


-------

## Weather Variables

For Höstvete & Bladfläckssvampar we have fetched 7 days forecast for all grading dates with weather parameters:



```
Sunshine_Duration_surface
Total_precipitation_surface_3_Hour_Accumulation
Precipitation_rate_surface_3_Hour_Average
Temperature_surface
Volumetric_Soil_Moisture_Content_depth_below_surface_layer
Wind_speed_gust_surface
Relative_humidity_height_above_ground
Specific_humidity_height_above_ground
Dewpoint_temperature_height_above_ground
Soil_temperature_depth_below_surface_layer
```






Since fetching the data iteratively is slow (several hours) we will do this concurrently with several threads & proxies.

The data

In [ ]:

%pip install siphon

import json
import random
import requests
from siphon.catalog import TDSCatalog
from siphon.ncss import NCSS
from datetime import datetime, timedelta
import xarray as xr
from io import BytesIO
import concurrent.futures
import time
import requests

QUERY_VARIABLES = ["Sunshine_Duration_surface","Wind_speed_gust_surface","Specific_humidity_height_above_ground"]
OUTPUT_BASES = ["sunshine", "wind", "humidity"]
FILE_NAMES = ["gfs_sweden_sunshine", "gfs_sweden_wind", "gfs_sweden_humidity"]
SKIP_UNTILS = [None, None, None]

#LOOP
for QUERY_VARIABLE, OUTPUT_BASE, FILE_NAME, SKIP_UNTIL in zip(QUERY_VARIABLES, OUTPUT_BASES, FILE_NAMES, SKIP_UNTILS):
  QUERY_VARIABLE = QUERY_VARIABLE
  PROGNOSIS = 336  #168 #hours ahead (168/24 = 7 days)
  OUTPUT_BASE = "drive/MyDrive/MLProject/weather/14-day/" + OUTPUT_BASE + "/"
  FILE_NAME = FILE_NAME #will auto append target grade date _YYYY-MM-DD
  SKIP_UNTIL = SKIP_UNTIL#"2023-08-09" #"2021-05-16"  #if something went wrong, pick up where left off


  UNIQUE_DATES_PATH = "/content/unchecked_unique_dates_for_bladfläcksvampar_all_crops.json"
  PROXIES_PATH = "/content/Webshare 100 proxies.txt"


  # GEOGRAPHIC BOUNDS FOR SWEDEN
  lat_min = 55.0
  lat_max = 70.0
  lon_min = 10.0
  lon_max = 25.0


  #load unique grading dates
  with open(UNIQUE_DATES_PATH, "r") as f:
      dates = json.load(f)


  if SKIP_UNTIL:
    tmp = []
    found = False
    for date in dates:
      if date == SKIP_UNTIL and not found:
        found = True
      if found == True:
        tmp.append(date)

    dates = tmp


  #load proxies from the text file, one per line.
  with open(PROXIES_PATH, "r") as f:
      #remove any empty lines and strip whitespace.
      proxies_data = [line.strip() for line in f if line.strip()]

  """
  # Build a list of proxy URLs using the first protocol listed for each proxy.
  proxy_list = []
  for proxy in proxies_data:
      protocol = proxy['protocols'][0]  # e.g., "socks5" or "socks4"
      ip = proxy['ip']
      port = proxy['port']
      proxy_url = f"{protocol}://{ip}:{port}"
      proxy_list.append(proxy_url)
  """

  #create a list to store each proxy's details.
  proxy_list = []
  protocol = "socks5"

  for proxy in proxies_data:
      parts = proxy.split(":")
      if len(parts) >= 4:
          ip, port, username, password = parts[:4]
          proxy_url = f"{protocol}://{username}:{password}@{ip}:{port}"
          proxy_entry = {
              "ip": ip,
              "port": port,
              "username": username,
              "password": password,
              "url": proxy_url
          }
          proxy_list.append(proxy_entry)

  print("Loaded proxies:", proxy_list)


  def download_dataset(file, catalog, proxy_list, lat_min, lat_max, lon_min, lon_max, timeout=10):
      """
      Downloads the dataset for the given file from the catalog using a randomly selected proxy.
      This function will keep retrying (with a default timeout and retry loop) until the data is successfully fetched.
      Returns an xarray.Dataset once successful.
      """
      dataset = catalog.datasets[file]
      if 'NetcdfSubset' not in dataset.access_urls:
          print(f"Dataset {file} doesn't support NetcdfSubset.")
          return None

      attempt = 0
      while True:
          attempt += 1
          try:
              #choose a random proxy from the list and use its URL string.
              selected_proxy = random.choice(proxy_list)
              proxies = {
                  'http': selected_proxy["url"],
                  'https': selected_proxy["url"],
              }
              #create a custom requests session with the proxy settings.
              session = requests.Session()
              session.proxies = proxies

              #setting a default timeout (by patching the original)
              original_request = session.request
              def request_with_timeout(method, url, **kwargs):
                  kwargs.setdefault('timeout', timeout)
                  return original_request(method, url, **kwargs)
              session.request = request_with_timeout

              #create an NCSS object and overwrite its session with our custom session.
              ncss_url = dataset.access_urls['NetcdfSubset']
              ncss = NCSS(ncss_url)
              ncss.session = session

              query = ncss.query()
              query.lonlat_box(north=lat_max, south=lat_min, east=lon_max, west=lon_min)
              query.variables(QUERY_VARIABLE)
              query.accept('netcdf4')

              print(f"Fetching data from {file} using proxy {selected_proxy['url']} (attempt {attempt})...")
              data = ncss.get_data(query)
              ds = xr.open_dataset(BytesIO(data))

              #no idea why they got time1 and time2 sometimes???
              if 'time1' in ds.coords:
                  ds = ds.rename({'time1': 'time'})
              if 'time2' in ds.coords:
                  ds = ds.rename({'time2': 'time'})

              return ds

          except Exception as e:
              if "Server Error ( 400:" in str(e):
                  return None

              print(f"Error fetching {file} (attempt {attempt}): {e}")
              #wait a sec before retrying
              time.sleep(1)


  for dateStr in dates:
      print(f"Processing date: {dateStr}")
      try:
          #subtract x amount of days from the given grading date to get the forecast data x days back up until the grading date
          format_str = '%Y-%m-%d'
          date_obj = datetime.strptime(dateStr, format_str) - timedelta(days=int(PROGNOSIS/24))
          yearStr = date_obj.strftime("%Y")
          dateStr7DaysBack = date_obj.strftime("%Y%m%d")
      except Exception as e:
          print(f"Error processing date {dateStr}: {e}")
          continue

      #construct the catalog URL.
      catalog_url = f'https://thredds.rda.ucar.edu/thredds/catalog/files/g/d084001/{yearStr}/{dateStr7DaysBack}/catalog.xml'
      print(f"\nProcessing date {dateStr} using catalog: {catalog_url}")

      try:
          catalog = TDSCatalog(catalog_url)
      except Exception as e:
          print(f"Error loading catalog for {dateStr}: {e}")
          continue

      #list and filter datasets.
      all_datasets = list(catalog.datasets.keys())
      init_time = dateStr7DaysBack + "12"
      prognosis = PROGNOSIS
      filtered_files = [f for f in all_datasets if init_time in f and int(f.split(".f")[-1][:3]) <= prognosis]
      filtered_files = [f for f in filtered_files if ".idx" not in f]
      #SKIP EVERYother FOR 3 HOUR AVG (RAIN)
      if QUERY_VARIABLE == "Precipitation_rate_surface_3_Hour_Average" or QUERY_VARIABLE == "Total_precipitation_surface_3_Hour_Accumulation":
        filtered_files = [file for i, file in enumerate(filtered_files) if i % 2 != 0]

      print("Selected files for processing:", filtered_files)
      ds_list = []

      #use ThreadPoolExecutor to download datasets concurrently (max 10 workers).
      with concurrent.futures.ThreadPoolExecutor(max_workers=10) as executor:
          #map each file to a concurrent download.
          future_to_file = {
              executor.submit(download_dataset, file, catalog, proxy_list, lat_min, lat_max, lon_min, lon_max): file
              for file in filtered_files
          }
          for future in concurrent.futures.as_completed(future_to_file):
              file = future_to_file[future]
              try:
                  ds = future.result()
                  if ds is not None:
                      ds_list.append(ds)
              except Exception as exc:
                  print(f"{file} generated an exception: {exc}")

      print("Number of datasets downloaded:", len(ds_list))
      if ds_list:
          try:
              #concatenate along the time dimension.
              ds_combined = xr.concat(ds_list, dim="time")
              print("Final dataset structure:")
              #print(ds_combined)

              output_filename = f"{OUTPUT_BASE}{FILE_NAME}_{dateStr}.nc"
              ds_combined.to_netcdf(output_filename)
              print(f"Saved dataset to {output_filename}")
          except Exception as e:
              print(f"Error concatenating or saving dataset for {dateStr}: {e}")
      else:
          print("No data was retrieved for this date. Please check dataset availability.")


Utdata för streaming har trunkerats till de sista 5000 raderna.
Server Error ( 503: Service Temporarily Unavailable)
Error fetching gfs.0p25.2020041712.f069.grib2 (attempt 1): Error accessing https://thredds.rda.ucar.edu/thredds/ncss/grid/files/g/d084001/2020/20200417/gfs.0p25.2020041712.f069.grib2?var=Total_precipitation_surface_3_Hour_Accumulation&west=10.0&east=25.0&south=55.0&north=70.0&accept=netcdf4
Server Error ( 503: Service Temporarily Unavailable)
Error fetching gfs.0p25.2020041712.f117.grib2 (attempt 1): Error accessing https://thredds.rda.ucar.edu/thredds/ncss/grid/files/g/d084001/2020/20200417/gfs.0p25.2020041712.f117.grib2?var=Total_precipitation_surface_3_Hour_Accumulation&west=10.0&east=25.0&south=55.0&north=70.0&accept=netcdf4
Server Error ( 503: Service Temporarily Unavailable)
Error fetching gfs.0p25.2020041712.f123.grib2 (attempt 1): Error accessing https://thredds.rda.ucar.edu/thredds/ncss/grid/files/g/d084001/2020/20200417/gfs.0p25.2020041712.f123.grib2/dataset.xm